# agent-app에 랭체인 적용

## prompt → ChatPromptTemplate
- System 메세지는 `app/prompts/answer_system.md` 파일로 작성해 두었다.
- md 파일을 템플릿에 넣으면 `{}` 가 변수로 인식되는 문제가 생길 수 있다.
- → `SystemMessage(content=md파일내용)` 으로 적용

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Tue" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W4" / "Tue"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 사내 규정 질의응답 도우미입니다."),
        ("human", "{question}")
    ]
)

# print("변수 목록 : ", prompt.input_variables)
value = prompt.invoke({"question": "부산 출장 일비는?"}) # 입력은 dict타입
print(value)

for msg in value.to_messages():
    print(msg.content)

In [ ]:
# prompt | llm | parser
from langchain_core.language_models import FakeListChatModel, ParrotFakeChatModel
from langchain_core.output_parsers import StrOutputParser

# 앵무새 모델
echo = (prompt | ParrotFakeChatModel()).invoke({"question": "부산 출장 일비는?"})
print(echo.content,  type(echo).__name__)

chain = (
    prompt
    | FakeListChatModel(responses=["부산 출장 일비는 1일 4만원입니다."])
    | StrOutputParser()
)
print("체인 결과 : ", chain.invoke({"question": "부산 출장 일비는?"}))
print("체인 타입 : ", type(chain).__name__)

In [ ]:
from pathlib import Path

PROMPT_PATH = Path("backend", "app", "prompts", "answer_system.md")
raw = PROMPT_PATH.read_text(encoding="utf-8")

print(f"answer_system.md : {len(raw.splitlines())} 줄, 중괄호 { raw.count('{') } 개 {raw.count('}')}개")

ChatPromptTemplate.from_messages([
    ("system", raw),
    ("human", "{question}")
]).invoke({"question": "부산 출장 일비는?"})
print("통과")

#------

risky = raw + '\n\n예시: {"answer": "....", "sources": [{"doc_id" : "DOC-HR-011"}]}'
try:
    ChatPromptTemplate.from_messages([
        ("system", risky),
        ("human", "{question}")
    ]).invoke({"question": "부산 출장 일비는?"})
except ValueError as e:
    print("예외 발생 ")
    print(e)

In [ ]:
from langchain_core.messages import SystemMessage

# 최종 버전
safe_prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=risky),
        ("human", "{question}")
    ]
)

val = safe_prompt.invoke({"question": "부산 출장 일비는?"})
print(val)
print(safe_prompt.input_variables)

## llm → LLMPort 를 체인에 넣을 수 있게 변환
- 우리 포트 규격: `answer(*, question, context, user) -> LLMResult`

In [ ]:
from langchain_core.runnables import Runnable, RunnableLambda

from app.integrations.ports import LLMPort, LLMResult

# ClaudeLLM, OpenAILLM 등...
class MockLLM:
    def answer(self, *, question: str, context: list[dict], user: dict) -> LLMResult:
        # 가짜 LLM의 가짜 응답 값 리턴
        return LLMResult(
            text="부산 출장 일비는 1일 4만원입니다.",
            model="mock",
            input_tok=10,
            output_tok=12,
            cost_krw=0.0,
            latency_ms=1
        )

print(isinstance(MockLLM(), LLMPort))

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=raw),
    ("human", "{question}")
])

llm = MockLLM()

# ChatPromptTemplate 받아 -> 우리 포트를 부르는 연결 고리
def call_port(value) -> LLMResult:
    msgs = value.to_messages()
    text = "\n\n".join(m.content for m in msgs)
    return llm.answer(question=text, context=[], user={})

# 체인에 연결 가능하게 runnable로 변환
llm_step = RunnableLambda(call_port).with_config(run_name="LLMPort")

chain = prompt | llm_step
result = chain.invoke({"question": "부산 출장 일비는?"})

print("result.text : ", result.text)
print(result.model)

## LLMResult 결과를 파서로 전달 → Runnable 결과 텍스트만 추출 (중간다리 역할)

In [ ]:
# chain = prompt | llm_step | RunnableLambda(lambda r: r.text)
# try:
#     (prompt | llm_step | StrOutputParser()).invoke({"question": "부산 출장 일비는?"})
# except Exception as e:
#     print("StrOutputParser 붙혔을때... ")
#     print(e)

text_chain = prompt | llm_step | RunnableLambda(lambda r: r.text)
answer_txt = text_chain.invoke({"question": "부산 출장 일비는?"})

print(answer_txt)

## 파서 자리 → PydanticOutputParser (검증)

```python
# 방법1 : 모델은 JSON을 달라는 말을 들은적이 없다 -> 비교적 좋지 못한 코드
parser = PydanticOutputParser(pydantic_object=AnswerOut)
chain = prompt | llm_step | RunnableLambda(lambda r: r.text) | parser

# 방법2 : 형식 지시를 프롬프트에 넣기 (answer_system.md 에 형식을 알려주고 있음. 우리는 partial은 생략가능)
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=raw),
    ("human", "{question}\n\n{format_instructions}")
]).partial(format_instructions=parser.get_format_instructions())
# partial : 변수 한칸을 미리 채우는 기능
```

In [ ]:
import json
from langchain_core.output_parsers import PydanticOutputParser
from app.schemas.chat import AnswerOut

parser = PydanticOutputParser(pydantic_object=AnswerOut)

print("형식 지시문 :", parser.get_format_instructions().splitlines()[0])

GOLDEN = {
    "answer": "부산 출장 일비는 1일 2만원입니다.",
    "sources": [
        {
            "doc_id": "DOC-HR-014",
            "title": "국내출장 여비 규정",
            "version": "v2.0",
            "locator": "제12조 · p.6",
        }
    ],
    "enough_evidence": True,
}

class MockLLMJson:
    def answer(self, *, question: str, context: list[dict], user: dict) -> LLMResult:
        return LLMResult(text=json.dumps(GOLDEN, ensure_ascii=False), model="mock")

llm_json = MockLLMJson()
llm_step_json = RunnableLambda(
    lambda value: llm_json.answer(
        question="\n\n".join(m.content for m in value.to_messages()),
        context=[],
        user={},
    )
).with_config(run_name="LLMPort")

parsed_chain = prompt | llm_step_json | RunnableLambda(lambda r: r.text) | parser
parsed = parsed_chain.invoke({"question": "부산 출장 일비는?"})

print("결과 타입 :", type(parsed).__name__)
print("   answer          :", parsed.answer)
print("   enough_evidence :", parsed.enough_evidence)

In [ ]:
class MockLLMPlain:
    def answer(self, *, question: str, context: list[dict], user: dict) -> LLMResult:
        return LLMResult(text="죄송합니다. 잘 모르겠습니다.", model="mock")

llm_plain = MockLLMPlain()
plain_chain = (
    prompt
    | RunnableLambda(
        lambda value: llm_plain.answer(
            question="\n\n".join(m.content for m in value.to_messages()),
            context=[],
            user={},
        )
    )
    | RunnableLambda(lambda r: r.text)
    | parser
)
try:
    plain_chain.invoke({"question": "부산 출장 일비는?"})
except Exception as exc:
    print("형식을 안 지켰을 때 :", type(exc).__name__)

## backend/app/agent/chain.py → 우리 프로젝트에 실제 적용

In [ ]:
import importlib

import app.agent.chain as chain_module

importlib.reload(chain_module)

from app.agent.chain import build_answer_chain, build_parsed_chain, load_prompt

system_prompt = load_prompt("answer_system")
print("프롬프트 앞부분 : ", system_prompt[:50])

text = build_answer_chain(MockLLM(), system_prompt=system_prompt).invoke(
    {"question": "부산 출장 일비는?"}
)
print("build answer chain : ", text)

parsed = build_parsed_chain(
    MockLLMJson(), schema=AnswerOut, system_prompt=system_prompt
).invoke({"question": "부산 출장 일비는?"})
print("parsed answer : ", parsed.answer)
print("parsed sources : ", parsed.sources[0].doc_id)